#**Labeling Manual data Berita**

In [ ]:
from google.colab import drive
import pandas as pd

# 1️⃣ Sambungkan Google Drive
drive.mount('/content/drive')

# 2️⃣ Path ke file CSV kamu
PATH = '/content/drive/MyDrive/Semester 7/PPW/berita_detik.csv'

# 3️⃣ Baca file
df = pd.read_csv(PATH)

# 4️⃣ Pastikan kolom kategori ada
if 'kategori' not in df.columns:
    df['kategori'] = 'Belum dikategorikan'

# 5️⃣ Definisi kata kunci per kategori
keywords = {
    'Finance': ['ihsg', 'rupiah', 'saham', 'investasi', 'bank', 'bursa', 'inflasi', 'ekonomi'],
    'Politik': ['presiden', 'menteri', 'politik', 'dpr', 'pemilu', 'partai', 'prabowo', 'jokowi'],
    'Olahraga': ['liga', 'piala', 'pertandingan', 'timnas', 'gol', 'fifa', 'sepak bola', 'persib', 'arema'],
    'Kesehatan': ['vaksin', 'covid', 'rumah sakit', 'virus', 'dokter', 'kesehatan', 'obat', 'bpjs'],
    'Makanan': ['resep', 'kuliner', 'restoran', 'makanan', 'minuman', 'menu', 'warung', 'kafe'],
}

# 6️⃣ Fungsi deteksi kategori
def detect_category(text):
    if not isinstance(text, str):
        return 'News'
    text_lower = text.lower()
    for kategori, kata_list in keywords.items():
        if any(kata in text_lower for kata in kata_list):
            return kategori
    return 'News'  # fallback ke News

# 7️⃣ Terapkan ke kolom judul + isi
df['kategori'] = df.apply(
    lambda row: detect_category(str(row['judul']) + ' ' + str(row['isi'])), axis=1
)

# 8️⃣ Simpan hasilnya ke file baru
output_path = '/content/drive/MyDrive/Semester 7/PPW/berita_detik_label_otomatis.csv'
df.to_csv(output_path, index=False)

print("✅ Labeling otomatis selesai! Semua yang tidak terdeteksi akan masuk kategori 'News'")
print("📁 Hasil disimpan di:", output_path)

# 9️⃣ Contoh hasil
print("\n🔍 Contoh hasil:")
print(df[['judul', 'kategori']].head(10))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Labeling otomatis selesai! Semua yang tidak terdeteksi akan masuk kategori 'News'
📁 Hasil disimpan di: /content/drive/MyDrive/Semester 7/PPW/berita_detik_label_otomatis.csv

🔍 Contoh hasil:
                                               judul   kategori
0  Alfamart Gelar Donor Darah Serentak di 34 Kota...  Kesehatan
1  Yusril: Presiden Tak Akan Bentuk Tim Investiga...    Politik
2  Bobby Nasution Wajibkan OPD di Sumut Beri Kete...  Kesehatan
3  Jelang Muktamar X, DPC PPP Se-Jateng Deklarasi...    Politik
4  Ada Diskon Tiket Whoosh untuk Keberangkatan 22...       News
5  Video: Profil Djamari Chaniago, Jenderal Tempu...       News
6  Jadi Kepala Badan Komunikasi Pemerintah, Angga...    Politik
7  Eropa Percepat Sanksi Energi Rusia di Tengah T...    Finance
8  Warga Segel Rumah Dapur MBG di Bandung karena ...       News
9  Dunia Hari Ini: Perjanjian Militer A

## **Prepocessing**

In [ ]:
# 1. Import library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
import joblib
import re


In [ ]:
pip install Sastrawi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 11.3 MB/s eta 0:00:00


**Input Data**

In [ ]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path ke file Excel (ganti sesuai lokasi file di Drive-mu)
file_path = "/content/drive/MyDrive/Semester 7/PPW/berita_detik_label_otomatis.csv"

# Baca file Excel
df = pd.read_csv(file_path)

# Tampilkan nama kolom
print("Nama-nama kolom:")
print(df.columns.tolist())

# Tampilkan 100 baris pertama
print("\nContoh 100 baris pertama:")
print(df.head(100))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nama-nama kolom:
['id', 'judul', 'link', 'kategori', 'isi']

Contoh 100 baris pertama:
     id                                              judul  \
0     1  Alfamart Gelar Donor Darah Serentak di 34 Kota...   
1     2  Yusril: Presiden Tak Akan Bentuk Tim Investiga...   
2     3  Bobby Nasution Wajibkan OPD di Sumut Beri Kete...   
3     4  Jelang Muktamar X, DPC PPP Se-Jateng Deklarasi...   
4     5  Ada Diskon Tiket Whoosh untuk Keberangkatan 22...   
..  ...                                                ...   
95   96  Warga Geruduk Rumah 'Bang Jago' yang Pukuli 2 ...   
96   97  MK Tak Terima Gugatan soal Standar Pendidikan ...   
97   98  KPK Panggil Dirut Taspen Terkait Kasus Investa...   
98   99  Erick Thohir Tiba di Istana Jelang Pelantikan ...   
99  100  Foto Trump-Epstein Mendadak Muncul di Kastil W...   

                                       

**Prepocessing**

In [ ]:
# Stopword lokal
stopword_indonesia = set([
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "karena",
    "ada", "saya", "kami", "kita", "mereka", "pada", "adalah", "juga", "tidak",
    "ya", "kok", "loh", "banget", "sih", "jadi", "udah", "lagi", "aja", "dong", "nih"
])

# Normalisasi kata
normalisasi_kata = {
    "bangeettt": "banget", "bgt": "banget", "bgtt": "banget",
    "gk": "tidak", "ga": "tidak", "nggak": "tidak",
    "dr": "dari", "tp": "tapi", "tdk": "tidak",
    "sy": "saya", "lg": "lagi"
}

def normalize_kata(tokens):
    return [normalisasi_kata.get(word, word) for word in tokens]

def simple_tokenize(text):
    return text.split()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = simple_tokenize(text)
    tokens = normalize_kata(tokens)
    tokens = [word for word in tokens if word not in stopword_indonesia and len(word) > 2]
    return " ".join(tokens)

# Terapkan preprocessing
df['preprocessed'] = df['isi'].astype(str).apply(preprocess)

In [ ]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Buat stemmer Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Ambil 5 data awal
sample_data = df['isi'].astype(str).head(5)

# List untuk simpan hasil
stopwords_removed = []
cleaning = []
normalized = []
stemming = []
tokenizing = []

for text in sample_data:
    # Tokenisasi awal (pisah kata)
    tokens = text.split()

    # Stopword removal
    tokens_sw = [word for word in tokens if word not in stopword_indonesia and len(word) > 2]
    stopwords_removed.append(tokens_sw)

    # Cleaning (hapus karakter non-huruf)
    tokens_clean = [re.sub(r'[^a-zA-Z]', '', word) for word in tokens_sw if re.sub(r'[^a-zA-Z]', '', word) != ""]
    cleaning.append(tokens_clean)

    # Normalisasi (pembakuan kata)
    tokens_norm = [normalisasi_kata.get(word.lower(), word.lower()) for word in tokens_clean]
    normalized.append(tokens_norm)

    # Stemming
    tokens_stem = [stemmer.stem(word) for word in tokens_norm]
    stemming.append(tokens_stem)

    # Tokenisasi akhir
    tokenizing.append(tokens_stem)

# Simpan hasil ke DataFrame
tahapan_df = pd.DataFrame({
    'Asli': sample_data,
    'Stopword Removal': stopwords_removed,
    'Cleaning': cleaning,
    'Normalisasi (Ejaan Baku)': normalized,
    'Stemming': stemming,
    'Tokenisasi Akhir': tokenizing
})

# Tampilkan hasil
from IPython.display import display
display(tahapan_df)


,Asli,Stopword Removal,Cleaning,Normalisasi (Ejaan Baku),Stemming,Tokenisasi Akhir
0,alfamart kembali menggelar aksi donor darah se...,"[alfamart, kembali, menggelar, aksi, donor, da...","[alfamart, kembali, menggelar, aksi, donor, da...","[alfamart, kembali, menggelar, aksi, donor, da...","[alfamart, kembali, gelar, aksi, donor, darah,...","[alfamart, kembali, gelar, aksi, donor, darah,..."
1,"menteri koordinator bidang hukum, ham, imigras...","[menteri, koordinator, bidang, hukum,, ham,, i...","[menteri, koordinator, bidang, hukum, ham, imi...","[menteri, koordinator, bidang, hukum, ham, imi...","[menteri, koordinator, bidang, hukum, ham, imi...","[menteri, koordinator, bidang, hukum, ham, imi..."
2,gubernur sumatera utara (sumut) bobby nasution...,"[gubernur, sumatera, utara, (sumut), bobby, na...","[gubernur, sumatera, utara, sumut, bobby, nasu...","[gubernur, sumatera, utara, sumut, bobby, nasu...","[gubernur, sumatera, utara, sumut, bobby, nasu...","[gubernur, sumatera, utara, sumut, bobby, nasu..."
3,dukungan untuk plt ketua umum partai persatuan...,"[dukungan, plt, ketua, umum, partai, persatuan...","[dukungan, plt, ketua, umum, partai, persatuan...","[dukungan, plt, ketua, umum, partai, persatuan...","[dukung, plt, ketua, umum, partai, satu, bangu...","[dukung, plt, ketua, umum, partai, satu, bangu..."
4,bagi pengguna whoosh rute jakarta-bandung atau...,"[bagi, pengguna, whoosh, rute, jakarta-bandung...","[bagi, pengguna, whoosh, rute, jakartabandung,...","[bagi, pengguna, whoosh, rute, jakartabandung,...","[bagi, guna, whoosh, rute, jakartabandung, ata...","[bagi, guna, whoosh, rute, jakartabandung, ata..."


In [ ]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Buat stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Ambil semua data (ubah ke string dulu)
sample_data = df['isi'].astype(str)

# List untuk menyimpan hasil
stopwords_removed = []
cleaning = []
normalized = []
stemming = []
tokenizing = []

for text in sample_data:
    # Tokenisasi awal
    tokens = text.split()

    # Stopword removal
    tokens_sw = [word for word in tokens if word.lower() not in stopword_indonesia and len(word) > 2]
    stopwords_removed.append(tokens_sw)

    # Cleaning (hapus karakter non-huruf)
    tokens_clean = [re.sub(r'[^a-zA-Z]', '', word) for word in tokens_sw if re.sub(r'[^a-zA-Z]', '', word) != ""]
    cleaning.append(tokens_clean)

    # Normalisasi (pembakuan kata)
    tokens_norm = [normalisasi_kata.get(word.lower(), word.lower()) for word in tokens_clean]
    normalized.append(tokens_norm)

    # Stemming
    tokens_stem = [stemmer.stem(word) for word in tokens_norm]
    stemming.append(tokens_stem)

    # Tokenisasi akhir
    tokenizing.append(tokens_stem)

# Gabungkan ke DataFrame
tahapan_df = pd.DataFrame({
    'Teks Asli': sample_data,
    'Setelah Stopword Removal': stopwords_removed,
    'Setelah Cleaning': cleaning,
    'Setelah Normalisasi': normalized,
    'Setelah Stemming': stemming,
    'Tokenisasi Akhir': tokenizing
})

# Tentukan lokasi penyimpanan di Google Drive
save_path = "/content/drive/MyDrive/Semester 7/PPW/hasil_preprocessing_berita.csv"

# Simpan otomatis ke Drive
tahapan_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ File berhasil disimpan di {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File berhasil disimpan di /content/drive/MyDrive/Semester 7/PPW/hasil_preprocessing_berita.csv
